# Uninformed Search Algorithms in Python
## Part A - Toy Graph and Bangladesh Highway Network

This notebook implements BFS, DFS, DLS, IDS, Bidirectional Search, and UCS for the given lab tasks. The same algorithms are applied to the toy tree and the Bangladesh highway graph.

In [1]:
print("Laiba Sumaiya Nazim 23549009041")

from collections import deque
import heapq

toy_graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F', 'G'],
    'D': ['B'],
    'E': ['B'],
    'F': ['C'],
    'G': ['C'],
}

toy_graph_weighted = {
    'A': [('B', 2), ('C', 5)],
    'B': [('A', 2), ('D', 4), ('E', 1)],
    'C': [('A', 5), ('F', 2), ('G', 3)],
    'D': [('B', 4)],
    'E': [('B', 1)],
    'F': [('C', 2)],
    'G': [('C', 3)],
}

bd_graph = {
    'Rangpur': ['Bogura', 'Mymensingh'],
    'Bogura': ['Rangpur', 'Rajshahi', 'Dhaka'],
    'Rajshahi': ['Bogura', 'Jessore'],
    'Jessore': ['Rajshahi', 'Khulna'],
    'Khulna': ['Jessore', 'Barisal', 'Dhaka'],
    'Barisal': ['Khulna', 'Dhaka'],
    'Mymensingh': ['Rangpur', 'Dhaka'],
    'Dhaka': ['Bogura', 'Khulna', 'Barisal', 'Mymensingh', 'Comilla'],
    'Comilla': ['Dhaka', 'Chittagong', 'Sylhet'],
    'Sylhet': ['Comilla'],
    'Chittagong': ['Comilla', "Cox's Bazar"],
    "Cox's Bazar": ['Chittagong'],
}

bd_graph_weighted = {
    'Rangpur': [('Bogura', 60), ('Mymensingh', 190)],
    'Bogura': [('Rangpur', 60), ('Rajshahi', 65), ('Dhaka', 210)],
    'Rajshahi': [('Bogura', 65), ('Jessore', 165)],
    'Jessore': [('Rajshahi', 165), ('Khulna', 55)],
    'Khulna': [('Jessore', 55), ('Barisal', 130), ('Dhaka', 210)],
    'Barisal': [('Khulna', 130), ('Dhaka', 165)],
    'Mymensingh': [('Rangpur', 190), ('Dhaka', 120)],
    'Dhaka': [('Bogura', 210), ('Khulna', 210), ('Barisal', 165), ('Mymensingh', 120), ('Comilla', 100)],
    'Comilla': [('Dhaka', 100), ('Chittagong', 150), ('Sylhet', 200)],
    'Sylhet': [('Comilla', 200)],
    'Chittagong': [('Comilla', 150), ("Cox's Bazar", 150)],
    "Cox's Bazar": [('Chittagong', 150)],
}

Laiba Sumaiya Nazim 23549009041


## Part A - Level 1: Toy Graph

In [2]:
def bfs(graph, start, goal):
    queue = deque([(start, [start])])
    visited = set()
    order = []

    while queue:
        node, path = queue.popleft()
        if node in visited:
            continue
        visited.add(node)
        order.append(node)

        if node == goal:
            return order, path

        for neighbor in graph[node]:
            if neighbor not in visited:
                queue.append((neighbor, path + [neighbor]))

    return order, None

bfs_order_toy, bfs_path_toy = bfs(toy_graph, 'A', 'G')
print("BFS visited order:", bfs_order_toy)
print("BFS path:", " -> ".join(bfs_path_toy))
print("Path length:", len(bfs_path_toy) - 1)

BFS visited order: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
BFS path: A -> C -> G
Path length: 2


In [3]:
def dfs_iterative(graph, start, goal):
    stack = [(start, [start])]
    visited = set()
    order = []

    while stack:
        node, path = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        order.append(node)

        if node == goal:
            return order, path

        for neighbor in reversed(graph[node]):
            if neighbor not in visited:
                stack.append((neighbor, path + [neighbor]))

    return order, None


def dfs_recursive(graph, start, goal):
    visited = set()
    order = []
    found_path = []

    def visit(node, path):
        visited.add(node)
        order.append(node)

        if node == goal:
            found_path.extend(path)
            return True

        for neighbor in graph[node]:
            if neighbor not in visited:
                if visit(neighbor, path + [neighbor]):
                    return True
        return False

    visit(start, [start])
    return order, found_path if found_path else None

dfs_i_order_toy, dfs_i_path_toy = dfs_iterative(toy_graph, 'A', 'G')
dfs_r_order_toy, dfs_r_path_toy = dfs_recursive(toy_graph, 'A', 'G')

print("DFS iterative visited order:", dfs_i_order_toy)
print("DFS iterative path:", " -> ".join(dfs_i_path_toy))
print()
print("DFS recursive visited order:", dfs_r_order_toy)
print("DFS recursive path:", " -> ".join(dfs_r_path_toy))
print()
print("Do the visited orders match?", dfs_i_order_toy == dfs_r_order_toy)
print("BFS visited order:", bfs_order_toy)
print("DFS visited order:", dfs_i_order_toy)

DFS iterative visited order: ['A', 'B', 'D', 'E', 'C', 'F', 'G']
DFS iterative path: A -> C -> G

DFS recursive visited order: ['A', 'B', 'D', 'E', 'C', 'F', 'G']
DFS recursive path: A -> C -> G

Do the visited orders match? True
BFS visited order: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
DFS visited order: ['A', 'B', 'D', 'E', 'C', 'F', 'G']


In [4]:
def dls(graph, start, goal, limit):
    order = []
    path = []

    def search(node, current_path, depth):
        order.append(node)

        if node == goal:
            path.extend(current_path)
            return "success"

        if depth == limit:
            return "cutoff"

        cutoff_found = False
        for neighbor in graph[node]:
            if neighbor not in current_path:
                result = search(neighbor, current_path + [neighbor], depth + 1)
                if result == "success":
                    return "success"
                if result == "cutoff":
                    cutoff_found = True

        return "cutoff" if cutoff_found else "failure"

    status = search(start, [start], 0)
    return status, order, path if path else None

for limit in [1, 2]:
    status, order, path = dls(toy_graph, 'A', 'G', limit)
    print(f"L = {limit}: status = {status}, visited = {order}")

print("Minimum depth limit needed to reach G: 2")

L = 1: status = cutoff, visited = ['A', 'B', 'C']
L = 2: status = success, visited = ['A', 'B', 'D', 'E', 'C', 'F', 'G']
Minimum depth limit needed to reach G: 2


In [5]:
def ids(graph, start, goal):
    total_visits = 0

    for limit in range(0, len(graph) + 1):
        status, order, path = dls(graph, start, goal, limit)
        total_visits += len(order)

        print(f"Limit {limit}: status = {status}, visits = {order}")

        if status == "success":
            return limit, total_visits, path

    return None, total_visits, None

ids_depth_toy, ids_visits_toy, ids_path_toy = ids(toy_graph, 'A', 'G')
print()
print("IDS found G at depth:", ids_depth_toy)
print("Total node visits:", ids_visits_toy)
print("IDS path:", " -> ".join(ids_path_toy))

Limit 0: status = cutoff, visits = ['A']
Limit 1: status = cutoff, visits = ['A', 'B', 'C']
Limit 2: status = success, visits = ['A', 'B', 'D', 'E', 'C', 'F', 'G']

IDS found G at depth: 2
Total node visits: 11
IDS path: A -> C -> G


In [6]:
def bidirectional_bfs(graph, start, goal):
    if start == goal:
        return [start], [start], [start], 0

    front = {start}
    back = {goal}
    front_parent = {start: None}
    back_parent = {goal: None}
    front_order = []
    back_order = []
    meeting = None

    while front and back and meeting is None:
        next_front = set()
        for node in front:
            front_order.append(node)
            for neighbor in graph[node]:
                if neighbor not in front_parent:
                    front_parent[neighbor] = node
                    if neighbor in back_parent:
                        meeting = neighbor
                        break
                    next_front.add(neighbor)
            if meeting:
                break
        if meeting:
            break
        front = next_front

        next_back = set()
        for node in back:
            back_order.append(node)
            for neighbor in graph[node]:
                if neighbor not in back_parent:
                    back_parent[neighbor] = node
                    if neighbor in front_parent:
                        meeting = neighbor
                        break
                    next_back.add(neighbor)
            if meeting:
                break
        back = next_back

    if meeting is None:
        return front_order, back_order, None, len(front_order) + len(back_order)

    left = []
    node = meeting
    while node is not None:
        left.append(node)
        node = front_parent[node]
    left.reverse()

    right = []
    node = back_parent[meeting]
    while node is not None:
        right.append(node)
        node = back_parent[node]

    path = left + right
    return front_order, back_order, path, len(front_order) + len(back_order)

front_toy, back_toy, bi_path_toy, bi_expanded_toy = bidirectional_bfs(toy_graph, 'A', 'G')
print("Forward expansion:", front_toy)
print("Backward expansion:", back_toy)
print("Meeting node:", bi_path_toy[len(bi_path_toy)//2])
print("Reconstructed path:", " -> ".join(bi_path_toy))
print("Total expanded nodes:", bi_expanded_toy)

Forward expansion: ['A']
Backward expansion: ['G']
Meeting node: C
Reconstructed path: A -> C -> G
Total expanded nodes: 2


In [7]:
def ucs(graph, start, goal):
    pq = [(0, start, [start])]
    best_cost = {start: 0}
    order = []

    while pq:
        cost, node, path = heapq.heappop(pq)

        if cost != best_cost.get(node):
            continue

        order.append(node)

        if node == goal:
            return order, path, cost

        for neighbor, weight in graph[node]:
            new_cost = cost + weight
            if new_cost < best_cost.get(neighbor, float("inf")):
                best_cost[neighbor] = new_cost
                heapq.heappush(pq, (new_cost, neighbor, path + [neighbor]))

    return order, None, None

ucs_order_toy, ucs_path_toy, ucs_cost_toy = ucs(toy_graph_weighted, 'A', 'G')
print("UCS visited order:", ucs_order_toy)
print("UCS path:", " -> ".join(ucs_path_toy))
print("Total cost:", ucs_cost_toy)

print()
print("On this tree, there is only one possible path from A to G.")
print("Therefore, UCS, BFS and DFS return the same path; UCS additionally calculates its total cost.")

UCS visited order: ['A', 'B', 'E', 'C', 'D', 'F', 'G']
UCS path: A -> C -> G
Total cost: 8

On this tree, there is only one possible path from A to G.
Therefore, UCS, BFS and DFS return the same path; UCS additionally calculates its total cost.


## Part A - Comparison and Cross-Edge Discussion

In [8]:
toy_summary = {
    "BFS": (len(bfs_order_toy), len(bfs_path_toy) - 1, bfs_path_toy),
    "DFS": (len(dfs_i_order_toy), len(dfs_i_path_toy) - 1, dfs_i_path_toy),
    "DLS (L=2)": (len(dls(toy_graph, 'A', 'G', 2)[1]), 2, dls(toy_graph, 'A', 'G', 2)[2]),
    "IDS": (ids_visits_toy, ids_depth_toy, ids_path_toy),
    "Bidirectional BFS": (bi_expanded_toy, len(bi_path_toy) - 1, bi_path_toy),
    "UCS": (len(ucs_order_toy), len(ucs_path_toy) - 1, ucs_path_toy),
}

for name, (visited, length, path) in toy_summary.items():
    print(f"{name}: visited/expanded = {visited}, path length = {length}, path = {' -> '.join(path)}")

print()
print("If D-F is added as a cross-edge, the graph is no longer a tree and it contains a cycle.")
print("BFS can still find a shortest-hop path, while DFS may choose a different route.")
print("Visited sets become important to prevent repeated visits and infinite loops.")
print("UCS can also choose a lower-cost path if multiple paths become available.")

BFS: visited/expanded = 7, path length = 2, path = A -> C -> G
DFS: visited/expanded = 7, path length = 2, path = A -> C -> G
DLS (L=2): visited/expanded = 7, path length = 2, path = A -> C -> G
IDS: visited/expanded = 11, path length = 2, path = A -> C -> G
Bidirectional BFS: visited/expanded = 2, path length = 2, path = A -> C -> G
UCS: visited/expanded = 7, path length = 2, path = A -> C -> G

If D-F is added as a cross-edge, the graph is no longer a tree and it contains a cycle.
BFS can still find a shortest-hop path, while DFS may choose a different route.
Visited sets become important to prevent repeated visits and infinite loops.
UCS can also choose a lower-cost path if multiple paths become available.


## Part B - Level 2: Bangladesh Highway Network

In [9]:
bd_bfs_order, bd_bfs_path = bfs(bd_graph, 'Rangpur', "Cox's Bazar")
print("BFS visited order:", bd_bfs_order)
print("BFS path:", " -> ".join(bd_bfs_path))
print("Hop count:", len(bd_bfs_path) - 1)

BFS visited order: ['Rangpur', 'Bogura', 'Mymensingh', 'Rajshahi', 'Dhaka', 'Jessore', 'Khulna', 'Barisal', 'Comilla', 'Chittagong', 'Sylhet', "Cox's Bazar"]
BFS path: Rangpur -> Bogura -> Dhaka -> Comilla -> Chittagong -> Cox's Bazar
Hop count: 5


In [10]:
bd_dfs_order, bd_dfs_path = dfs_iterative(bd_graph, 'Rangpur', "Cox's Bazar")
print("DFS visited order:", bd_dfs_order)
print("DFS path:", " -> ".join(bd_dfs_path))
print("Hop count:", len(bd_dfs_path) - 1)
print("BFS hop count:", len(bd_bfs_path) - 1)

DFS visited order: ['Rangpur', 'Bogura', 'Rajshahi', 'Jessore', 'Khulna', 'Barisal', 'Dhaka', 'Mymensingh', 'Comilla', 'Chittagong', "Cox's Bazar"]
DFS path: Rangpur -> Bogura -> Rajshahi -> Jessore -> Khulna -> Barisal -> Dhaka -> Comilla -> Chittagong -> Cox's Bazar
Hop count: 9
BFS hop count: 5


In [11]:
for limit in [2, 3, 4, 5]:
    status, order, path = dls(bd_graph, 'Rangpur', "Cox's Bazar", limit)
    print(f"L = {limit}: status = {status}, visits = {len(order)}, path = {path}")

L = 2: status = cutoff, visits = 6, path = None
L = 3: status = cutoff, visits = 15, path = None
L = 4: status = cutoff, visits = 27, path = None
L = 5: status = success, visits = 19, path = ['Rangpur', 'Bogura', 'Dhaka', 'Comilla', 'Chittagong', "Cox's Bazar"]


In [12]:
bd_ids_depth, bd_ids_visits, bd_ids_path = ids(bd_graph, 'Rangpur', "Cox's Bazar")
print()
print("IDS depth:", bd_ids_depth)
print("IDS total visits:", bd_ids_visits)
print("IDS path:", " -> ".join(bd_ids_path))
print("Same depth as BFS?", bd_ids_depth == len(bd_bfs_path) - 1)
print("Same path as BFS?", bd_ids_path == bd_bfs_path)

Limit 0: status = cutoff, visits = ['Rangpur']
Limit 1: status = cutoff, visits = ['Rangpur', 'Bogura', 'Mymensingh']
Limit 2: status = cutoff, visits = ['Rangpur', 'Bogura', 'Rajshahi', 'Dhaka', 'Mymensingh', 'Dhaka']
Limit 3: status = cutoff, visits = ['Rangpur', 'Bogura', 'Rajshahi', 'Jessore', 'Dhaka', 'Khulna', 'Barisal', 'Mymensingh', 'Comilla', 'Mymensingh', 'Dhaka', 'Bogura', 'Khulna', 'Barisal', 'Comilla']
Limit 4: status = cutoff, visits = ['Rangpur', 'Bogura', 'Rajshahi', 'Jessore', 'Khulna', 'Dhaka', 'Khulna', 'Jessore', 'Barisal', 'Barisal', 'Khulna', 'Mymensingh', 'Comilla', 'Chittagong', 'Sylhet', 'Mymensingh', 'Dhaka', 'Bogura', 'Rajshahi', 'Khulna', 'Jessore', 'Barisal', 'Barisal', 'Khulna', 'Comilla', 'Chittagong', 'Sylhet']
Limit 5: status = success, visits = ['Rangpur', 'Bogura', 'Rajshahi', 'Jessore', 'Khulna', 'Barisal', 'Dhaka', 'Dhaka', 'Khulna', 'Jessore', 'Rajshahi', 'Barisal', 'Barisal', 'Khulna', 'Jessore', 'Mymensingh', 'Comilla', 'Chittagong', "Cox's Bazar

In [13]:
bd_front, bd_back, bd_bi_path, bd_bi_expanded = bidirectional_bfs(
    bd_graph, 'Rangpur', "Cox's Bazar"
)
print("Forward expansion:", bd_front)
print("Backward expansion:", bd_back)
print("Meeting city:", bd_bi_path[len(bd_bi_path)//2])
print("Reconstructed path:", " -> ".join(bd_bi_path))
print("Bidirectional expanded nodes:", bd_bi_expanded)
print("BFS visited nodes:", len(bd_bfs_order))

Forward expansion: ['Rangpur', 'Mymensingh', 'Bogura', 'Rajshahi', 'Dhaka']
Backward expansion: ["Cox's Bazar", 'Chittagong']
Meeting city: Comilla
Reconstructed path: Rangpur -> Mymensingh -> Dhaka -> Comilla -> Chittagong -> Cox's Bazar
Bidirectional expanded nodes: 7
BFS visited nodes: 12


In [14]:
bd_ucs_order, bd_ucs_path, bd_ucs_cost = ucs(
    bd_graph_weighted, 'Rangpur', "Cox's Bazar"
)
print("UCS visited order:", bd_ucs_order)
print("Least-distance path:", " -> ".join(bd_ucs_path))
print("Total distance:", bd_ucs_cost, "km")

UCS visited order: ['Rangpur', 'Bogura', 'Rajshahi', 'Mymensingh', 'Dhaka', 'Jessore', 'Khulna', 'Comilla', 'Barisal', 'Chittagong', 'Sylhet', "Cox's Bazar"]
Least-distance path: Rangpur -> Bogura -> Dhaka -> Comilla -> Chittagong -> Cox's Bazar
Total distance: 670 km


In [15]:
bd_summary = {
    "BFS": (len(bd_bfs_order), len(bd_bfs_path) - 1, bd_bfs_path),
    "DFS": (len(bd_dfs_order), len(bd_dfs_path) - 1, bd_dfs_path),
    "DLS (L=5)": (len(dls(bd_graph, 'Rangpur', "Cox's Bazar", 5)[1]), len(dls(bd_graph, 'Rangpur', "Cox's Bazar", 5)[2]) - 1, dls(bd_graph, 'Rangpur', "Cox's Bazar", 5)[2]),
    "IDS": (bd_ids_visits, bd_ids_depth, bd_ids_path),
    "Bidirectional BFS": (bd_bi_expanded, len(bd_bi_path) - 1, bd_bi_path),
    "UCS": (len(bd_ucs_order), len(bd_ucs_path) - 1, bd_ucs_path),
}

for name, (visited, length, path) in bd_summary.items():
    print(f"{name}: visited/expanded = {visited}, path length = {length}, path = {' -> '.join(path)}")

print()
print("Note: BFS and IDS use the unweighted graph, so they optimize the number of hops.")
print("UCS uses the weighted graph, so it optimizes total highway distance.")

BFS: visited/expanded = 12, path length = 5, path = Rangpur -> Bogura -> Dhaka -> Comilla -> Chittagong -> Cox's Bazar
DFS: visited/expanded = 11, path length = 9, path = Rangpur -> Bogura -> Rajshahi -> Jessore -> Khulna -> Barisal -> Dhaka -> Comilla -> Chittagong -> Cox's Bazar
DLS (L=5): visited/expanded = 19, path length = 5, path = Rangpur -> Bogura -> Dhaka -> Comilla -> Chittagong -> Cox's Bazar
IDS: visited/expanded = 71, path length = 5, path = Rangpur -> Bogura -> Dhaka -> Comilla -> Chittagong -> Cox's Bazar
Bidirectional BFS: visited/expanded = 7, path length = 5, path = Rangpur -> Mymensingh -> Dhaka -> Comilla -> Chittagong -> Cox's Bazar
UCS: visited/expanded = 12, path length = 5, path = Rangpur -> Bogura -> Dhaka -> Comilla -> Chittagong -> Cox's Bazar

Note: BFS and IDS use the unweighted graph, so they optimize the number of hops.
UCS uses the weighted graph, so it optimizes total highway distance.


In [18]:
print("Laiba Sumaiya Nazim 23549009041")
def path_cost(graph, path):
    cost = 0
    for i in range(len(path) - 1):
        current_node = path[i]
        next_node = path[i+1]
        for neighbor, weight in graph[current_node]:
            if neighbor == next_node:
                cost += weight
                break
    return cost
def depth_limited_search(graph, current, goal, depth, path):
    if current == goal:
        return path

    if depth == 0:
        return None
    for neighbor, cost in graph[current]:
        if neighbor not in path:
            result = depth_limited_search(
                graph, neighbor, goal, depth - 1, path + [neighbor]
            )
            if result is not None:
                return result
    return None
def iterative_deepening_search(graph, start, goal, max_depth=20):
    for depth in range(max_depth + 1):
        result_path = depth_limited_search(
            graph, start, goal, depth, [start]
        )

        if result_path is not None:
            return result_path
    return None
ids_path = iterative_deepening_search(
    bd_graph_weighted, 'Rangpur', "Cox's Bazar"
)
ids_cost = None
if ids_path:
    ids_cost = path_cost(bd_graph_weighted, ids_path)

print("IDS Path:", ids_path)
print("IDS Total Distance:", ids_cost)

Laiba Sumaiya Nazim 23549009041
IDS Path: ['Rangpur', 'Bogura', 'Dhaka', 'Comilla', 'Chittagong', "Cox's Bazar"]
IDS Total Distance: 670


In [21]:
print("Laiba Sumaiya Nazim 23549009041")
import heapq
def reconstruct_path(parent, start, goal):
    path = []
    current = goal
    while current != start:
        path.append(current)
        current = parent[current]
    path.append(start)
    return path[::-1]
def uniform_cost_search(graph, start, goal):
    frontier = [(0, start)]
    parent = {start: None}
    cost_so_far = {start: 0}
    expanded = []
    while frontier:
        current_cost, current = heapq.heappop(frontier)
        if current_cost > cost_so_far.get(current, float('inf')):
            continue
        expanded.append(current)
        if current == goal:
            path = reconstruct_path(parent, start, goal)
            return path, current_cost, expanded
        for neighbor, edge_cost in graph[current]:
            new_cost = current_cost + edge_cost
            if new_cost < cost_so_far.get(neighbor, float("inf")):
                cost_so_far[neighbor] = new_cost
                parent[neighbor] = current
                heapq.heappush(
                    frontier,
                    (new_cost, neighbor)
                )
    return [], float("inf"), expanded
ucs_path, ucs_cost, ucs_expanded = uniform_cost_search(
    bd_graph_weighted,
    'Rangpur',
    "Cox's Bazar"
)
print("UCS Path:", ucs_path)
print("UCS Total Distance:", ucs_cost)
print("UCS Expanded Nodes:", ucs_expanded)
print("Number of Expanded Nodes:", len(ucs_expanded))

Laiba Sumaiya Nazim 23549009041
UCS Path: ['Rangpur', 'Bogura', 'Dhaka', 'Comilla', 'Chittagong', "Cox's Bazar"]
UCS Total Distance: 670
UCS Expanded Nodes: ['Rangpur', 'Bogura', 'Rajshahi', 'Mymensingh', 'Dhaka', 'Jessore', 'Khulna', 'Comilla', 'Barisal', 'Chittagong', 'Sylhet', "Cox's Bazar"]
Number of Expanded Nodes: 12


## Conclusion

The experiments show how different uninformed search algorithms behave on a simple tree and a connected highway network. BFS is useful for finding the shallowest route in an unweighted graph, while DFS can find a route without guaranteeing the fewest hops. DLS and IDS control the search depth, Bidirectional Search searches from both ends, and UCS uses edge costs to find the least-distance route. The results also show why graph cycles require careful use of visited sets.